# Lesson 1: Router Engine

Welcome to Lesson 1.

To access the `requirements.txt` file, the data/pdf file required for this lesson and the `helper` and `utils` modules, please go to the `File` menu and select`Open...`.

I hope you enjoy this course!

## Setup

In [1]:
from helper import get_openai_api_key

OPENAI_API_KEY = get_openai_api_key()

In [2]:
import nest_asyncio

nest_asyncio.apply()

## Load Data

To download this paper, below is the needed code:

#!wget "https://openreview.net/pdf?id=VtmBAGCN7o" -O metagpt.pdf

**Note**: The pdf file is included with this lesson. To access it, go to the `File` menu and select`Open...`.

In [6]:
from llama_index.core import SimpleDirectoryReader

# load documents
documents = SimpleDirectoryReader(input_files=["metagpt.pdf"]).load_data()

In [9]:
type(documents)
len(documents)
documents[0]

Document(id_='7104cc8f-2eec-4807-95ab-c6e6b8ec64e9', embedding=None, metadata={'page_label': '1', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2026-08-03', 'last_modified_date': '2024-06-24'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, text='Preprint\nMETAGPT: M ETA PROGRAMMING FOR A\nMULTI -AGENT COLLABORATIVE FRAMEWORK\nSirui Hong1∗, Mingchen Zhuge2∗, Jonathan Chen1, Xiawu Zheng3, Yuheng Cheng4,\nCeyao Zhang4,Jinlin Wang1,Zili Wang ,Steven Ka Shing Yau5,Zijuan Lin4,\nLiyang Zhou6,Chenyu Ran1,Lingfeng Xiao1,7,Chenglin Wu1†,J¨urgen Schmidhuber2,8\n1DeepWisdom,2AI Initiative, King Abdullah University of Science and Technology,\n3Xiamen University,4The Chinese University of 

## Define LLM and Embedding model

In [10]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)

In [11]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAI(model="gpt-3.5-turbo")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")

## Define Summary Index and Vector Index over the Same Data

In [12]:
from llama_index.core import SummaryIndex, VectorStoreIndex

summary_index = SummaryIndex(nodes)
vector_index = VectorStoreIndex(nodes)

## Define Query Engines and Set Metadata

In [13]:
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True,
)
vector_query_engine = vector_index.as_query_engine()

In [14]:
from llama_index.core.tools import QueryEngineTool


summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description=(
        "Useful for summarization questions related to MetaGPT"
    ),
)

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description=(
        "Useful for retrieving specific context from the MetaGPT paper."
    ),
)

In [18]:
type(vector_tool)

llama_index.core.tools.query_engine.QueryEngineTool

## Define Router Query Engine

In [19]:
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector


query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[
        summary_tool,
        vector_tool,
    ],
    verbose=True
)

In [20]:
response = query_engine.query("What is the summary of the document?")
print(str(response))

Selecting query engine 0: Useful for summarization questions related to MetaGPT.
The document introduces MetaGPT, a meta-programming framework that enhances the problem-solving capabilities of multi-agent systems using Large Language Models (LLMs). MetaGPT assigns specific roles to agents, streamlines workflows, and incorporates human-like Standardized Operating Procedures (SOPs) to improve task decomposition. It emphasizes structured outputs, a communication protocol, role specialization, workflow management, and efficient sharing mechanisms. The framework achieves state-of-the-art performance in code generation benchmarks, offers a robust solution for developing LLM-based multi-agent systems, and inspires future research on human-inspired techniques for artificial multi-agent systems.


In [21]:
print(len(response.source_nodes))

34


In [22]:
response = query_engine.query(
    "How do agents share information with other agents?"
)
print(str(response))

Selecting query engine 1: This choice is more relevant as it specifically mentions retrieving specific context, which would be necessary to understand how agents share information with other agents..
Agents share information with other agents by utilizing a shared message pool where they can publish structured messages. This shared message pool allows all agents to exchange messages directly, enabling them to both publish their own messages and access messages from other agents transparently. Additionally, agents can subscribe to relevant messages based on their role profiles, allowing them to extract the information they need for their specific tasks and responsibilities.


In [26]:
response = query_engine.query(
    " Write a summary"
)
print(str(response))

Selecting query engine 0: The first choice is specifically mentioned as being useful for summarization questions related to MetaGPT, which aligns with the task of writing a summary..
MetaGPT is a meta-programming framework that enhances multi-agent collaboration through human-like Standard Operating Procedures (SOPs) based on Large Language Models (LLMs). It assigns roles to agents, streamlines workflows, and improves task decomposition, ensuring efficient collaboration with structured outputs and a communication protocol. The framework excels in code generation benchmarks, offering a robust solution for developing LLM-based multi-agent systems. MetaGPT models agents as a simulated software company, emphasizing role specialization, workflow management, and efficient communication mechanisms. It incorporates an executable feedback mechanism for real-time code generation quality improvement and has demonstrated superior performance in various benchmarks. The integration of human-like SOP

## Let's put everything together

In [23]:
from utils import get_router_query_engine

query_engine = get_router_query_engine("metagpt.pdf")

In [24]:
response = query_engine.query("Tell me about the ablation study results?")
print(str(response))

Selecting query engine 1: Ablation study results are specific context from the MetaGPT paper, making choice 2 the most relevant..
The ablation study results demonstrate the effectiveness of MetaGPT in addressing challenges related to information overload and reducing hallucinations in software generation. By utilizing a global message pool and a subscription mechanism, MetaGPT successfully streamlines communication to enhance relevance and utility of information, while also guiding thinking in granular tasks to prevent code hallucination issues commonly faced by large language models.
